# Task
Fine-tune a GPT-2 model on a custom text dataset using the `transformers` and `datasets` libraries. This involves installing dependencies like `torch` and `accelerate`, loading the pre-trained GPT-2 model and tokenizer, preparing the dataset for causal language modeling, training the model using the `Trainer` API, and generating text to evaluate the performance.

## Install Dependencies



In [1]:
!pip install -q transformers datasets torch accelerate
print("Libraries installed successfully.")

Libraries installed successfully.


## Load Model and Tokenizer

### Subtask:
Load the pre-trained GPT-2 model and its corresponding tokenizer from the Hugging Face Hub and configure padding.


In [2]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

# Load pre-trained GPT-2 model and tokenizer
model_name = 'gpt2'
model = GPT2LMHeadModel.from_pretrained(model_name)
tokenizer = GPT2Tokenizer.from_pretrained(model_name)

# GPT-2 does not have a pad token, so we use the eos_token
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id

print(f"Model and tokenizer '{model_name}' loaded and padding configured.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Model and tokenizer 'gpt2' loaded and padding configured.


In [3]:
import warnings
from transformers import GPT2LMHeadModel, GPT2Tokenizer, logging

# Suppress non-critical warnings and load reports from transformers
logging.set_verbosity_error()
warnings.filterwarnings("ignore", message=".*HF_TOKEN.*")
warnings.filterwarnings("ignore", message=".*unexpected keys.*")

# Load pre-trained GPT-2 model and tokenizer
model_name = 'gpt2'
model = GPT2LMHeadModel.from_pretrained(model_name)
tokenizer = GPT2Tokenizer.from_pretrained(model_name)

# GPT-2 does not have a pad token, so we use the eos_token
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id

print(f"Model and tokenizer '{model_name}' loaded and padding configured successfully.")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Model and tokenizer 'gpt2' loaded and padding configured successfully.


## Prepare Custom Dataset

### Subtask:
Create a small custom text dataset and prepare it for fine-tuning the GPT-2 model for causal language modeling.


In [4]:
from datasets import Dataset

# 1. Create a simple list of strings related to AI
data = [
    "Artificial Intelligence is transforming the world of technology.",
    "GPT-2 is a transformer-based language model developed by OpenAI.",
    "Large language models can generate human-like text based on prompts.",
    "Machine learning involves training algorithms on large datasets.",
    "Fine-tuning allows pre-trained models to adapt to specific domains."
]

# 2. Convert list to a Hugging Face Dataset object
raw_dataset = Dataset.from_dict({"text": data})

# 3 & 4. Define tokenization function with labels for causal LM
def tokenize_function(examples):
    outputs = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )
    # For Causal LM, labels are typically a copy of input_ids
    outputs["labels"] = outputs["input_ids"].copy()
    return outputs

# 5. Apply tokenization to the dataset
tokenized_dataset = raw_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=raw_dataset.column_names
)

# 6. Inspect a single sample
sample = tokenized_dataset[0]
print("Keys in tokenized dataset:", sample.keys())
print(f"Input IDs length: {len(sample['input_ids'])}")
print(f"First 10 input_ids: {sample['input_ids'][:10]}")
print(f"First 10 labels: {sample['labels'][:10]}")

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

Keys in tokenized dataset: dict_keys(['input_ids', 'attention_mask', 'labels'])
Input IDs length: 128
First 10 input_ids: [8001, 9542, 9345, 318, 25449, 262, 995, 286, 3037, 13]
First 10 labels: [8001, 9542, 9345, 318, 25449, 262, 995, 286, 3037, 13]


## Fine-tune GPT-2

### Subtask:
Configure the training arguments and use the Trainer API to fine-tune the GPT-2 model on the tokenized dataset.


In [8]:
from transformers import TrainingArguments, Trainer

# 1. Define the training configuration without the problematic arguments
training_args = TrainingArguments(
    output_dir="./gpt2-finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    save_steps=10,
    save_total_limit=2,
    logging_steps=1,
    prediction_loss_only=True,
    report_to="none"
)

# 2. Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
)

# 3. Start the fine-tuning process
print("Starting fine-tuning with corrected arguments...")
trainer.train()
print("Fine-tuning completed successfully.")

Starting fine-tuning with corrected arguments...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'loss': '8.949', 'grad_norm': '197.7', 'learning_rate': '5e-05', 'epoch': '0.3333'}
{'loss': '5.764', 'grad_norm': '164.7', 'learning_rate': '4.444e-05', 'epoch': '0.6667'}
{'loss': '3.782', 'grad_norm': '139.3', 'learning_rate': '3.889e-05', 'epoch': '1'}
{'loss': '1.989', 'grad_norm': '116', 'learning_rate': '3.333e-05', 'epoch': '1.333'}
{'loss': '1.338', 'grad_norm': '86.13', 'learning_rate': '2.778e-05', 'epoch': '1.667'}
{'loss': '1.407', 'grad_norm': '87.05', 'learning_rate': '2.222e-05', 'epoch': '2'}
{'loss': '0.7524', 'grad_norm': '44.22', 'learning_rate': '1.667e-05', 'epoch': '2.333'}
{'loss': '0.5401', 'grad_norm': '23.02', 'learning_rate': '1.111e-05', 'epoch': '2.667'}
{'loss': '1.122', 'grad_norm': '59.71', 'learning_rate': '5.556e-06', 'epoch': '3'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '72.23', 'train_samples_per_second': '0.208', 'train_steps_per_second': '0.125', 'train_loss': '2.849', 'epoch': '3'}
Fine-tuning completed successfully.


**Reasoning**:
With the model successfully fine-tuned on the custom AI dataset, the final step is to evaluate its performance by generating text from a prompt to verify that it has learned to produce domain-specific content.



In [9]:
import torch

# 1. Set the model to evaluation mode
model.eval()

# 2. Define an AI-related starting prompt
prompt = "Artificial Intelligence is"

# 3. Tokenize input and move to the model's device
inputs = tokenizer(prompt, return_tensors="pt")
device = next(model.parameters()).device
inputs = {k: v.to(device) for k, v in inputs.items()}

# 4. Generate text using sampling for varied and coherent output
print("Generating text based on fine-tuned model...")
with torch.no_grad():
    output_sequences = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=50,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        no_repeat_ngram_size=2,
        pad_token_id=tokenizer.pad_token_id
    )

# 5. Decode and display the result
generated_text = tokenizer.decode(output_sequences[0], skip_special_tokens=True)
print("\n--- Generated Output ---")
print(generated_text)

Generating text based on fine-tuned model...

--- Generated Output ---
Artificial Intelligence is not a threat.
